# Make an AI funnier with one vector

**A hands-on intro to activation steering, the idea behind abliteration. Made for anyone new.**

Inside a language model, ideas like *humor*, *formality* or *refusal* live as **directions** in its activations.
Find the direction, add it while the model runs, and the behavior changes. No retraining, no fine-tuning.

In about 3 minutes you will:
1. see the numbers a model "thinks" in,
2. find the humor direction inside a real model,
3. turn it up and down and watch the answers change,
4. make your own direction: pirate, Shakespeare, grumpy, anything.

**How to run:** `Runtime → Run all`. A free T4 GPU is already selected, nothing to install.

By Abhinav Vashisht · [explainer thread on X](https://x.com/abhi_inthewild/status/2113974169113370766)

In [ ]:
#@title 1. Load a small open model (about a minute)
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"  #@param ["Qwen/Qwen2.5-1.5B-Instruct", "Qwen/Qwen2.5-0.5B-Instruct", "Qwen/Qwen2.5-3B-Instruct"]

import html, textwrap
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=dtype).to(device).eval()
layers = model.model.layers
print(f"{MODEL} on {device}: {len(layers)} layers, {model.config.hidden_size} numbers per word")
if device == "cpu":
    print("No GPU found. It still works, just slower. In Colab: Runtime > Change runtime type > T4 GPU")

def show(*panels):
    """Show (title, text) pairs side by side."""
    try:
        from IPython.display import HTML, display
        cards = "".join(
            f'<div style="flex:1;min-width:240px;border:1px solid #8884;border-radius:10px;padding:12px">'
            f'<div style="font-weight:600;margin-bottom:6px">{html.escape(title)}</div>'
            f'<div style="white-space:pre-wrap;line-height:1.45">{html.escape(text)}</div></div>'
            for title, text in panels)
        display(HTML(f'<div style="display:flex;gap:12px;flex-wrap:wrap">{cards}</div>'))
    except ImportError:
        for title, text in panels:
            print(title.upper() + "\n" + textwrap.fill(text, 100) + "\n")

## 2. Every word becomes a list of numbers

As the model reads your prompt, each word becomes a long list of numbers, and every layer updates that list.
This list is the model's **activation**: what it is "thinking" at that point.

In [ ]:
#@title Look at one activation
def chat(prompt, system=None):
    msgs = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": prompt}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def activations(prompt, system=None):
    """The model's numbers at the last word of the prompt, one list per layer."""
    ids = tok(chat(prompt, system), return_tensors="pt").to(device)
    with torch.no_grad():
        hidden = model(**ids, output_hidden_states=True).hidden_states
    return torch.stack([h[0, -1].float() for h in hidden[1:]])  # [layers, numbers]

acts = activations("Why is the sky blue?")
mid = len(layers) // 2
print(f"Layer {mid}, last word of the prompt: {acts.shape[1]} numbers. The first ten:")
print(acts[mid, :10].cpu().numpy().round(2))

## 3. Find the humor direction

First the model writes two answers to each question: a funny one and a serious one.
Then it reads each answer back while we record its activations.
Average the funny ones, average the serious ones, subtract, and what is left is the **humor direction**:
the pattern that changes when the text is funny. This takes about a minute.

In [ ]:
#@title Compare funny vs serious answers
QUESTIONS = [
    "How do I make coffee?", "Why is the sky blue?", "Give me tips for a job interview.",
    "What should I cook tonight?", "How does a computer work?", "How do I learn to swim?",
    "What is a black hole?", "How do I save money?", "Explain how rain works.", "Why do we sleep?",
]
FUNNY = "You are a hilarious comedian. Answer the question, but make it full of jokes, puns and silly exaggeration."
SERIOUS = "You are a dry, serious, formal assistant. Answer precisely. Never joke."

def write_answers(system, questions):
    answers = []
    for q in questions:
        ids = tok(chat(q, system), return_tensors="pt").to(device)
        out = model.generate(**ids, max_new_tokens=48, do_sample=False, pad_token_id=tok.eos_token_id)
        answers.append(tok.decode(out[0, ids["input_ids"].shape[1]:], skip_special_tokens=True))
    return answers

def read_activations(question, answer):
    """Average activation over the answer's words, one list per layer."""
    start = len(tok(chat(question))["input_ids"])
    ids = tok(chat(question) + answer, return_tensors="pt").to(device)
    with torch.no_grad():
        hidden = model(**ids, output_hidden_states=True).hidden_states
    return torch.stack([h[0, start:].float().mean(0) for h in hidden[1:]])

def find_direction(positive, negative, questions=QUESTIONS, example=True):
    pos_answers, neg_answers = write_answers(positive, questions), write_answers(negative, questions)
    if example:
        show(("+ " + questions[0], pos_answers[0] + "..."), ("- " + questions[0], neg_answers[0] + "..."))
    pos = torch.stack([read_activations(q, a) for q, a in zip(questions, pos_answers)]).mean(0)
    neg = torch.stack([read_activations(q, a) for q, a in zip(questions, neg_answers)]).mean(0)
    return pos - neg  # one direction per layer

humor = find_direction(FUNNY, SERIOUS)
print(f"Found the humor direction: {humor.shape[0]} layers x {humor.shape[1]} numbers")

## 4. Add it while the model runs

Now we ask a normal question with **no system prompt**. Both answers use the same random seed,
so the only difference is the vector we add inside the model.

The **humor meter** measures how far each answer points along the humor direction.
That is the whole idea in one number: the further the activations point that way, the funnier the model acts.

Move the slider and run the cell again. Around 2 to 3 the answer gets playful.
Past 4 it starts rambling, and higher still it breaks into nonsense, which is fun to see too.
Negative values make it dry and formal.

In [ ]:
#@title Try it
PROMPT = "Explain how rain works."  #@param {type:"string"}
HUMOR = 2.5  #@param {type:"slider", min:-4, max:8, step:0.5}

DEFAULT_LAYER = int(len(layers) * 0.45)

def steer(direction, strength, layer):
    vec = (strength * direction[layer]).to(device, dtype)
    def hook(module, inputs, output):
        if isinstance(output, tuple):
            return (output[0] + vec,) + tuple(output[1:])
        return output + vec
    return layers[layer].register_forward_hook(hook)

def generate(prompt, direction=None, strength=0.0, layer=None, max_new_tokens=120, seed=0):
    layer = DEFAULT_LAYER if layer is None else layer
    handle = steer(direction, strength, layer) if direction is not None and strength else None
    try:
        torch.manual_seed(seed)
        ids = tok(chat(prompt), return_tensors="pt").to(device)
        out = model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=True, temperature=0.7,
                             top_p=0.9, repetition_penalty=1.1, pad_token_id=tok.eos_token_id)
        return tok.decode(out[0, ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    finally:
        if handle:
            handle.remove()

def meter(prompt, answer, direction, layer=None):
    """How far the answer's activations point along the direction."""
    layer = DEFAULT_LAYER if layer is None else layer
    unit = direction[layer] / direction[layer].norm()
    return float(read_activations(prompt, answer)[layer] @ unit)

normal = generate(PROMPT)
funny = generate(PROMPT, humor, HUMOR)
show(("normal", normal), (f"+ humor x {HUMOR}", funny))
gap = meter(PROMPT, funny, humor) - meter(PROMPT, normal, humor)
print(f"humor meter: the steered answer points {gap:+.1f} further along the humor direction than the normal one")

### Turn the dial

Same question, four strengths, side by side.

In [ ]:
#@title The dial
DIAL_PROMPT = "Why do cats sleep so much?"  #@param {type:"string"}
show(*[(f"humor x {s}", generate(DIAL_PROMPT, humor, s, max_new_tokens=70)) for s in (-3, 0, 2.5, 5)])

## 5. Make your own direction

Pick a preset, or choose `custom` and write your own pair of opposites.
The notebook finds the new direction the same way it found humor, then steers with it.

Try **grumpy**: the model starts refusing to help. Refusal is just another direction,
which is exactly the one abliteration finds and removes.

In [ ]:
#@title Your own direction
PRESET = "pirate"  #@param ["pirate", "shakespeare", "grumpy", "custom"]
CUSTOM_POSITIVE = "You always talk like a cowboy."  #@param {type:"string"}
CUSTOM_NEGATIVE = "You talk like a plain, normal assistant."  #@param {type:"string"}
STRENGTH = 4  #@param {type:"slider", min:-4, max:8, step:0.5}
TEST_PROMPT = "How do I make coffee?"  #@param {type:"string"}

PLAIN = "You talk like a plain, normal assistant."
PRESETS = {
    "pirate": "You always talk like a pirate.",
    "shakespeare": "You always speak like a character in a Shakespeare play.",
    "grumpy": "You are extremely grumpy and annoyed about everything.",
}
positive, negative = (CUSTOM_POSITIVE, CUSTOM_NEGATIVE) if PRESET == "custom" else (PRESETS[PRESET], PLAIN)
mine = find_direction(positive, negative, example=False)
show(("normal", generate(TEST_PROMPT)), (f"+ {PRESET} x {STRENGTH}", generate(TEST_PROMPT, mine, STRENGTH)))

## What you just did

1. **Every word became a list of numbers.** That list is the model's activation.
2. **An idea turned out to be a direction** in that list: humor, pirate, Shakespeare, even refusal.
3. **Adding the direction changed the behavior**, and the meter showed how far the answer pointed along it.

## How this connects to abliteration

Abliteration is the same idea in reverse. It finds the **refusal** direction by comparing prompts the model refuses
with prompts it answers, then **removes** it from every layer and bakes that into the weights.
The model keeps everything else and simply stops saying no.
That is why researchers worry about it, and why this notebook only adds harmless directions.

Full explainer with diagrams: [thread on X](https://x.com/abhi_inthewild/status/2113974169113370766)

**Further reading**
- Turner et al. (2023), [Activation Addition](https://arxiv.org/abs/2308.10248)
- Panickssery et al. (2023), [Steering Llama 2 via Contrastive Activation Addition](https://arxiv.org/abs/2312.06681)
- Zou et al. (2023), [Representation Engineering](https://arxiv.org/abs/2310.01405)
- Arditi et al. (2024), [Refusal in Language Models Is Mediated by a Single Direction](https://arxiv.org/abs/2406.11717)
- Anthropic, [Golden Gate Claude](https://www.anthropic.com/news/golden-gate-claude) and [Persona vectors](https://www.anthropic.com/research/persona-vectors)